In [1]:
#***************** Importation de données de Reddit *******************

import praw
import datetime
from RedditDocument import RedditDocument

docs = [] # contient les objets reddit

def fetch_reddit_posts(client_id, client_secret, user_agent, subreddit_name, nbposts=10):
    reddit = praw.Reddit(   # Connection to Reddit API
        client_id=client_id,
        client_secret=client_secret,
        user_agent=user_agent
    )

    subreddit = reddit.subreddit(subreddit_name) # specification du sujet

    for post in subreddit.hot(limit=nbposts): # the hottest nbposts
        readable_date = datetime.datetime.fromtimestamp( # conversion date
            post.created_utc
        ).strftime('%Y-%m-%d %H:%M:%S')

        doc = RedditDocument( # instanciation classe RedditDocument 
            titre=post.title,
            auteur=str(post.author),
            date=readable_date,
            url=post.url,
            texte=post.selftext,
            nb_comments=post.num_comments
        )

        docs.append(doc)  # save the added doc

    return docs



In [2]:
#***************** Importation de données de Arxiv *******************

import urllib.request as libreq
import xmltodict
from ArxivDocument import ArxivDocument

def fetch_arxiv(theme="machine+learning", max_results=10):
    url = f"http://export.arxiv.org/api/query?search_query=all:{theme}&start=0&max_results={max_results}"

    # Required USER AGENT
    req = libreq.Request(
        url,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Python Script'}
    )

    data = libreq.urlopen(req).read()
    feed = xmltodict.parse(data)

    entries = feed["feed"].get("entry", [])

    # If single entry, wrap into list
    if not isinstance(entries, list):
        entries = [entries]

    docsA = []

    for entry in entries:
        title = entry["title"].strip()

        authors = (
            [a["name"] for a in entry["author"]]
            if isinstance(entry["author"], list)
            else [entry["author"]["name"]]
        )

        date = entry["published"]
        text = entry["summary"]

        url = None
        links = entry.get("link", [])
        if isinstance(links, dict):
            links = [links]
        for l in links:
            if l["@rel"] == "alternate":  # main HTML page
                url = l["@href"]

        doc = ArxivDocument(title, authors, date, url, text) # instanciation classe ArxivDocument
        docsA.append(doc)

    return docsA


In [3]:
from Corpus import Corpus
from RedditDocument import RedditDocument
from ArxivDocument import ArxivDocument

# import fetch_reddit_posts
# import fetch_arxiv

In [4]:
import nltk
# nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

In [5]:
# main.py
from Corpus import Corpus


def main():
    corpus = Corpus("Search_Engine")

    # --- REDDIT ---
    reddit_data = fetch_reddit_posts(
        client_id="MYp9TRuDb4U_nHdmnuvL9Q",
        client_secret="FH1vXNJyBg_wxrQPsOohQqM5wuR1XA",
        user_agent="WebScraping",
        subreddit_name="MachineLearning",
        nbposts=50
    )

    for d in reddit_data:
        doc = Corpus.factory(
            texte=d.texte,
            titre=d.titre,
            auteur=d.auteur,
            date=d.date,
            source="reddit",
            url=d.url,
            nb_comments=d.getNbComments()
        )
        corpus.add_document(doc)


    # --- ARXIV ---
    arxiv_data = fetch_arxiv("MachineLearning", max_results=50)

    for d in arxiv_data:
        doc = Corpus.factory(
            texte=d.texte,
            titre=d.titre,
            auteur=d.auteur,
            date=d.date,
            source="arxiv",
            url=d.url
        )
        corpus.add_document(doc)

    # SAVE
    corpus.save("corpus.csv")

    print("DONE:", corpus)


if __name__ == "__main__":
    main()


Corpus saved to corpus.csv (72 documents).
DONE: <Corpus 'Search_Engine' — 72 docs / 65 auteurs>


In [6]:
import pandas as pd

# df = pd.read_csv("corpus.csv", sep=";", encoding="utf-8")
# print(df.columns)

# corpus = Corpus.load("corpus.csv", name="Search_Engine")
# print("Documents loaded:", len(corpus.id2doc))


In [7]:
# TEST SEARCH
# matches = corpus.search("machine learning")
# print(f"{len(matches)} documents found:")

# for m in matches:  # show first 3 matches
#     print(f"ID: {m['id']}\nAuteur: {m['auteur']}\nTitre: {m['titre']}\nURL: {m['url']}")
#     print(m['texte'][:100])  # preview first 100 characters
#     print("-----\n")


In [8]:
# df = corpus.concorde("workflow", context_size=50)

# print(f"{len(df)} matches found:\n")

# # display nicely
# for _, row in df.iterrows():  # iterate over DataFrame rows
#     print(f"ID: {row['id']}\nAuteur: {row['auteur']}\nTitre: {row['titre']}\nURL: {row['url']}")
#     # add ellipses to show left/right context
#     print("..." + row['texte'] + "...")
#     print("-----\n")

In [9]:
# df = corpus.stats(n=10, stop_words=stop_words)
# print(df)

In [10]:
# --- Example usage ---

#TEST TFIDF

# Set of stopwords
# stop_words = set(stopwords.words('english'))

# Initialize your corpus (assuming it's already populated)
# Example: corpus = Corpus.load("corpus.csv", name="Search_Engine")
# or add documents manually: corpus.add_document(doc)

# 1️⃣ Build vocabulary
# vocab_dict = corpus.build_vocab_dict(stop_words=stop_words)
# print("Vocabulary sample (first 10 words with IDs):")
# for i, (w, info) in enumerate(vocab_dict.items()):
#     if i >= 10:
#         break
#     print(f"{w}: {info}")

# 2️⃣ Build TF matrix
# mat_TF, vocab_dict = corpus.build_TF_matrix(stop_words=stop_words)
# print(f"\nTF matrix shape: {mat_TF.shape}")  # (number of docs x vocab size)

# 3️⃣ Update vocab statistics
# vocab_dict = corpus.update_vocab_stats(mat_TF, vocab_dict)
# print("\nVocabulary with total occurrences and document frequency (first 10):")
# for i, (w, info) in enumerate(vocab_dict.items()):
#     if i >= 10:
#         break
#     print(f"{w}: TF={info['total_occurrences']}, DF={info['document_frequency']}")

# 4️⃣ Build TF-IDF matrix
# mat_TF_IDF = corpus.build_TF_IDF_matrix(mat_TF, vocab_dict)
# print(f"\nTF-IDF matrix shape: {mat_TF_IDF.shape}")

# 5️⃣ Show top words by term frequency using stats()
# freq_df = corpus.stats(n=10, stop_words=stop_words)
# print("\nTop 10 words by term frequency:")
# print(freq_df)


In [11]:
# query = "workflow "
# results = corpus.search_engine(query, top_k=5)

# for r in results:
#     print(f"ID: {r['id']}\nAuteur: {r['auteur']}\nTitre: {r['titre']}\nURL: {r['url']}")
#     print(f"Score: {r['score']:.4f}")
#     print(f"Text preview: {r['texte'][:200]}...\n-----\n")


In [12]:
# TEST SEARCH

# # 1) Check vocab size
# print("vocab built?", bool(corpus.vocab), "len:", len(corpus.vocab))

# # 2) If empty, build it:
# from nltk.corpus import stopwords
# stop_words = set(stopwords.words('english'))
# corpus.vocab = corpus.build_vocab_dict(stop_words=stop_words)
# print("vocab size after build:", len(corpus.vocab))

# # 3) Build TF matrix and TF-IDF explicitly (optional)
# mat_TF, vocab_dict = corpus.build_TF_matrix(stop_words=stop_words)
# corpus.mat_TF = mat_TF
# corpus.vocab = vocab_dict
# corpus.vocab = corpus.update_vocab_stats(corpus.mat_TF, corpus.vocab)
# corpus.mat_TFXIDF = corpus.build_TF_IDF_matrix(corpus.mat_TF, corpus.vocab)

# # 4) Inspect whether the cleaned query token is in vocab:
# q = corpus.nettoyer_texte("workflow").split()
# print("clean tokens:", q, "in vocab:", [t for t in q if t in corpus.vocab])


In [13]:
# stop_words = set(stopwords.words('english'))
# corpus.build_all(stop_words=stop_words)
# corpus.search_engine("workflow", debug=True)


In [14]:
# import pandas as pd
# from Corpus import Corpus

# # 1️⃣ Load corpus from CSV
# csv_file = "corpus.csv"  # replace with your CSV path
# corpus = Corpus.load(csv_file, name="TestCorpus")

# print(corpus)  # print summary

# # 2️⃣ Build TF and TF-IDF matrices
# stop_words = set()  # or use nltk stopwords
# corpus.vocab = corpus.build_vocab_dict(stop_words=stop_words)
# corpus.mat_TF, _ = corpus.build_TF_matrix(stop_words=stop_words)
# corpus.mat_TFXIDF = Corpus.build_TF_IDF_matrix(corpus.mat_TF, corpus.vocab)

# print(f"Vocabulary size: {len(corpus.vocab)}")
# print(f"TF matrix shape: {corpus.mat_TF.shape}")
# print(f"TF-IDF matrix shape: {corpus.mat_TFXIDF.shape}")



In [15]:
# hethi l search b fonc l corpus
# # 3️⃣ Run a test search query
# corpus = Corpus.load("corpus.csv", name="Search_Engine")
# # print("Documents loaded:", len(corpus.id2doc))
# engine = SearchEngine(corpus, stop_words=stop_words, use_tfidf=True)

# query = "machine"
# results = corpus.search_engine(query, top_k=5, use_tfidf=True, stop_words=stop_words)

# # 4️⃣ Display results as DataFrame
# df_results = pd.DataFrame(results)
# print(df_results[["id", "titre", "auteur", "score"]])


In [16]:
# search bl engine
# 1️⃣ Load corpus

from SearchEngine import SearchEngine

corpus = Corpus.load("corpus.csv", name="Search_Engine")

# 2️⃣ Create search engine
engine = SearchEngine(corpus, stop_words=stop_words, use_tfidf=True)

# 3️⃣ Run search query
query = "machine learning"
results = engine.search(query, k=5)

# 4️⃣ Display results
print(results[["doc_id", "titre", "auteur", "score", "texte"]])


Corpus loaded from corpus.csv (72 documents).
SearchEngine initialized:
- vocab size: 3012
- matrix shape: (72, 3012)
   doc_id                                              titre  \
0      59  Tokamak disruption prediction using different ...   
1      38      [D] Amazon Applied Scientist 1 Interview loop   
2      51  From Cutting Planes Algorithms to Compression ...   
3      62   Deep Learning for Musculoskeletal Image Analysis   
4      41  [P] AI Learns to Play StarFox (Snes) (Deep Rei...   

               auteur     score  \
0       Joost Croonen  0.226155   
1  Realistic_Tea_2798  0.179159   
2      Liva Ralaivola  0.137339   
3      Ismail Irmakci  0.133355   
4   AgeOfEmpires4AOE4  0.104091   

                                               texte  
0  disruption prediction and mitigation is of key...  
1  hi everyone hope all of you are doing great th...  
2  provide a natural framework to perform machine...  
3  the diagnosis prognosis and treatment of patie...  
4  this tra

In [17]:
# # !!!!!!!!!!!!!! ask prof
# is it okay for arxiv docs 5alit auteur we7d eli houwa loul ma5alitch liste 
# i removed stop words